In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
from sklearn.metrics import r2_score
import os 
import glob
import math
import scipy.signal
import pwlf
import warnings
import re

class Test:
    def __init__(self, df):
        # Create a copy of the input DataFrame to preserve the original data
        self.df = df.copy()  
        
        # Extract participant ID and test date from the DataFrame for reference
        self.id = df['participant_ID'].iloc[0]  # Store participant ID
        self.date = df['visit_date'].iloc[0]    # Store test date
        
        # Generate a unique test identifier using participant ID and test date
        self.name = f"{self.id}_{self.date}"    

        # Initialize 'Stage' column and assign stage numbers only during the 'EXERCISE' phase
        self.df['Stage'] = np.nan
        exercise_mask = self.df['Phase'] == 'EXERCISE'
        
        # Group by 'Speed' and 'Grade' and assign stage numbers within the 'EXERCISE' phase
        self.df.loc[exercise_mask, 'Stage'] = self.df[exercise_mask].groupby(['Speed', 'Grade']).ngroup() + 1
        # Fill in the 'Stage' column with 0 for all rows that are not in the 'EXERCISE' phase
        self.df['Stage'] = self.df['Stage'].fillna(0)  

        # Convert 't' column to timedelta format and set it as the index of the DataFrame
        self.df['t'] = pd.to_timedelta(self.df['t'])
        self.df.set_index('t', inplace=True)

        # Prepare lactate-related data for further analysis
        self.lactate_df = self.prepare_lactate_df()
    
    def prepare_lactate_df(self):
        """
        Prepare the lactate data by:
        - Dropping rows with missing lactate values
        - Filtering out unrealistic lactate values (> 20)
        - Applying a logarithmic transformation to lactate and speed
        """
        # Drop rows where lactate values are missing and reset the index
        lactate_df = self.df.dropna(subset=['La-']).reset_index()
        lactate_df = lactate_df[['t', 'La-', 'Phase', 'Speed', 'Grade', 'Stage']]
        
        # Filter out physiologically implausible lactate values (greater than 20)
        lactate_df = lactate_df[lactate_df['La-'] <= 20]
        
        # Log-transform lactate and speed values (handle zero values gracefully)
        lactate_df['log_lactate'] = np.log(lactate_df['La-'].replace(0, np.nan))
        lactate_df['log_speed'] = np.log(lactate_df['Speed'].replace(0, np.nan))
        
        return lactate_df

    def lt_ref_vals(self, plot=False):
        """
        Identify lactate threshold 1 (LT1) and 2 (LT2) based on relative increases in lactate.
        - LT1 is the first significant increase in lactate (>= 0.5 mmol/L)
        - LT2 is the next significant increase (>= 1 mmol/L)
        Returns the lactate values and speeds at these thresholds.
        """
        # Identify the first significant increase in lactate (>= 0.5 mmol/L) for LT1
        lt1ref_index = (self.lactate_df['La-'].diff() >= 0.5)
        
        # Identify the next significant increase in lactate (>= 1 mmol/L) for LT2
        lt2ref_index = (self.lactate_df['La-'].diff() >= 1)
        
        # Retrieve lactate values and speeds at the first occurrence of LT1 and LT2
        LT1ref = self.lactate_df.loc[lt1ref_index.idxmax(), 'La-'] if lt1ref_index.any() else None
        LT2ref = self.lactate_df.loc[lt2ref_index.idxmax(), 'La-'] if lt2ref_index.any() else None
        LT1ref_speed, LT1ref_grade = self.lactate_df.loc[lt1ref_index.idxmax(), ['Speed', 'Grade']] if lt1ref_index.any() else (None, None)
        LT2ref_speed, LT2ref_grade = self.lactate_df.loc[lt2ref_index.idxmax(), ['Speed', 'Grade']] if lt2ref_index.any() else None
        
        if plot:
            valid_lactate = self.lactate_df.dropna(subset=['La-'])
            time_seconds = valid_lactate['t'].dt.total_seconds()
            time_labels = time_seconds.apply(lambda x: f"{int(x // 60):02}:{int(x % 60):02}")
            plt.figure(figsize=(10, 6))
            plt.plot(time_seconds, valid_lactate['La-'], label='Lactate vs Time', color='blue')
            plt.axhline(y=LT1ref, color='green', label='LT1')
            plt.axhline(y=LT2ref, color='red', linestyle='--', label='LT2')
            plt.xlabel('Time')
            plt.ylabel('Lactate (mmol/L)')
            plt.title('Lactate Thresholds (Reference Method)')
            plt.xticks(time_seconds, time_labels, rotation=45)
            plt.legend()
            plt.grid()
            plt.show()

        # Return lactate threshold values and corresponding speeds
        results = {
            "LT1_ref": LT1ref,
            "LT2_ref": LT2ref,
            "LT1_ref_speed": LT1ref_speed,
            "LT2_ref_speed": LT2ref_speed,
            "LT1_ref_grade": LT1ref_grade,
            "LT2_ref_grade": LT2ref_grade
        }
        return results

    def lt_abs_vals(self, plot=False):
        """
        Identify lactate thresholds using absolute lactate values:
        - LT1 is defined as 2 mmol/L
        - LT2 is defined as 4 mmol/L
        Returns the lactate values and speeds at these thresholds.
        """
        # Identify lactate values greater than or equal to 2 mmol/L for LT1
        lt1abs_index = (self.lactate_df['La-'] >= 2)
        
        # Identify lactate values greater than or equal to 4 mmol/L for LT2
        lt2abs_index = (self.lactate_df['La-'] >= 4)
        
        # Retrieve lactate values and speeds at the first occurrence of LT1 and LT2
        LT1abs = self.lactate_df.loc[lt1abs_index.idxmax(), 'La-'] if lt1abs_index.any() else None
        LT2abs = self.lactate_df.loc[lt2abs_index.idxmax(), 'La-'] if lt2abs_index.any() else None
        LT1abs_speed, LT1abs_grade = self.lactate_df.loc[lt1abs_index.idxmax(), ['Speed', 'Grade']] if lt1abs_index.any() else None
        LT2abs_speed, LT2abs_grade = self.lactate_df.loc[lt2abs_index.idxmax(), ['Speed','Grade']] if lt2abs_index.any() else None
        
        # Plot the lactate data with thresholds
        if plot:
            valid_lactate = self.lactate_df.dropna(subset=['La-'])
            time_seconds = valid_lactate['t'].dt.total_seconds()
            time_labels = time_seconds.apply(lambda x: f"{int(x // 60):02}:{int(x % 60):02}")
            plt.figure(figsize=(10, 6))
            plt.plot(time_seconds, valid_lactate['La-'], label='Lactate vs Speed', color='blue')
            plt.axhline(y=LT1abs, color='green', label='LT1 (2 mmol/L)')
            plt.axhline(y=LT2abs, color='red', linestyle='--', label='LT2 (4 mmol/L)')
            plt.xlabel('Time')
            plt.ylabel('Lactate (mmol/L)')
            plt.title('Lactate Thresholds (Absolute Method)')
            plt.xticks(time_seconds, time_labels, rotation=45)
            plt.legend()
            plt.grid()
            plt.show()

        # Return lactate threshold values and corresponding speeds
        results = {
            "LT1_abs": LT1abs,
            "LT2_abs": LT2abs,
            "LT1_abs_speed": LT1abs_speed,
            "LT2_abs_speed": LT2abs_speed,
            "LT1_abs_grade": LT1abs_grade,
            "LT2_abs_grade": LT2abs_grade
        }
        return results

    def v_slope_method(self, x_list, y_list):
        """
        Identifies the breakpoint (lactate threshold) in a curve using the V-slope method.
        The method uses two linear regressions to identify the breakpoint in lactate versus speed data.
        Returns:
        - The index of the breakpoint
        - Slopes and intercepts for both regressions
        - The intersection point (lactate threshold)
        """
        # Initial guess for the best split point in the data
        split_index = len(x_list) // 2
        best_index = split_index
        min_error = float('inf')  # Start with infinite error
        
        # Iterate over potential split points (avoiding the edges of the data)
        for i in range(10, len(x_list) - 10):
            # Perform linear regression for both halves of the data
            slope1, intercept1, _, _, _ = linregress(x_list[:i], y_list[:i])
            slope2, intercept2, _, _, _ = linregress(x_list[i:], y_list[i:])
            
            # Ensure that the slope increases after the breakpoint
            if slope1 < 1 and slope2 >= 1:
                # Calculate squared error for both fits
                error = np.sum((y_list[:i] - (slope1 * x_list[:i] + intercept1))**2) + \
                        np.sum((y_list[i:] - (slope2 * x_list[i:] + intercept2))**2)
                # Update the best split if the error is smaller
                if error < min_error:
                    min_error = error
                    best_index = i
        
        # Perform final regression fits using the best identified split
        slope1, intercept1, _, _, _ = linregress(x_list[:best_index], y_list[:best_index])
        slope2, intercept2, _, _, _ = linregress(x_list[best_index:], y_list[best_index:])
        
        # Calculate the intersection point (lactate threshold)
        x_threshold = (intercept2 - intercept1) / (slope1 - slope2)
        y_threshold = slope1 * x_threshold + intercept1
        
        # Return results: best split index, regression parameters, and intersection point
        return best_index, slope1, intercept1, slope2, intercept2, x_threshold, y_threshold

    def lt_log_semilog(self, plot=False):
        """
        Calculate lactate thresholds using logarithmic and semilogarithmic methods.
        - LT1: log(speed) vs log(lactate)
        - LT2: speed vs log(lactate)
        
        Optionally, plot the data and fitted lines.
        Returns a dictionary with the calculated lactate thresholds and R-squared values.
        """
        
        # === Prepare Data ===
        filtered_df = self.lactate_df.drop_duplicates(subset='Speed', keep='first')
        filtered_df = filtered_df[(filtered_df['Speed'] > 0) & (filtered_df['La-'] > 0)]

        log_speed = np.log(filtered_df['Speed'].values)
        log_lactate = np.log(filtered_df['La-'].values)
        speed = filtered_df['Speed'].values

        # === LT1: log(speed) vs log(lactate) ===
        my_pwlf1 = pwlf.PiecewiseLinFit(log_speed, log_lactate)
        breaks1 = my_pwlf1.fit(2)  # Fit two segments
        x_intersect_lt1 = breaks1[1]
        y_intersect_lt1 = my_pwlf1.predict([x_intersect_lt1])[0]
        lt1_log = math.exp(y_intersect_lt1)
        lt1_log_speed = math.exp(x_intersect_lt1)
        lt1_log_r2 = my_pwlf1.r_squared()

        # === LT2: speed vs log(lactate) ===
        my_pwlf2 = pwlf.PiecewiseLinFit(speed, log_lactate)
        breaks2 = my_pwlf2.fit(2)
        x_intersect_lt2 = breaks2[1]
        y_intersect_lt2 = my_pwlf2.predict([x_intersect_lt2])[0]
        lt2_semilog = x_intersect_lt2
        lt2_semilog_speed = math.exp(y_intersect_lt2)
        lt2_semi_log_r2 = my_pwlf2.r_squared()

        # === Plotting ===
        if plot:
            fig, axs = plt.subplots(1, 2, figsize=(12, 5))

            # LT1 plot
            axs[0].scatter(log_speed, log_lactate, color='gray', label='Data')
            x_hat1 = np.linspace(min(log_speed), max(log_speed), 100)
            y_hat1 = my_pwlf1.predict(x_hat1)
            axs[0].plot(x_hat1, y_hat1, color='blue', label='pwlf fit')
            axs[0].scatter(x_intersect_lt1, y_intersect_lt1, color='black', zorder=3, label='LT1')
            axs[0].set_title('LT1: Log Speed vs Log Lactate')
            axs[0].set_xlabel('Log Speed')
            axs[0].set_ylabel('Log Lactate')
            axs[0].legend()
            axs[0].grid()

            # LT2 plot
            axs[1].scatter(speed, log_lactate, color='gray', label='Data')
            x_hat2 = np.linspace(min(speed), max(speed), 100)
            y_hat2 = my_pwlf2.predict(x_hat2)
            axs[1].plot(x_hat2, y_hat2, color='red', label='pwlf fit')
            axs[1].scatter(x_intersect_lt2, y_intersect_lt2, color='black', zorder=3, label='LT2')
            axs[1].set_title('LT2: Speed vs Log Lactate')
            axs[1].set_xlabel('Speed')
            axs[1].set_ylabel('Log Lactate')
            axs[1].legend()
            axs[1].grid()

            plt.tight_layout()
            plt.show()

        # Return lactate threshold values and R-squared values
        return {
            "LT1_log":  lt1_log,
            "LT1_log_speed": lt1_log_speed,
            "LT1_log_r2": lt1_log_r2,
            "LT2_semilog": lt2_semilog, 
            "LT2_semilog_speed": lt2_semilog_speed,
            "LT2_semi_log_r2": lt2_semi_log_r2
        }
    
    def vt1_vt2_vo2_vco2(self, plot=False):
        vo2 = self.df['VO2']
        vco2 = self.df['VCO2']
        ve = self.df['VE']
        
        # Define an initial split index to separate the two regression regions
        split_index = len(vo2) // 2
        
        # Function to find the optimal intersection point
        best_index = split_index
        min_error = float('inf')
        
        for i in range(10, len(vo2) - 10):  # Ensure enough points for both regressions
            slope1, intercept1, _, _, _ = linregress(vo2[:i], vco2[:i])
            slope2, intercept2, _, _, _ = linregress(vo2[i:], vco2[i:])
            
            if slope1 < 1 and slope2 >= 1:  # Condition for V-slope method
                error = np.sum((vco2[:i] - (slope1 * vo2[:i] + intercept1))**2) + \
                        np.sum((vco2[i:] - (slope2 * vo2[i:] + intercept2))**2)
                if error < min_error:
                    min_error = error
                    best_index = i
        
        # Compute best fit lines
        slope1, intercept1, _, _, _ = linregress(vo2[:best_index], vco2[:best_index])
        slope2, intercept2, _, _, _ = linregress(vo2[best_index:], vco2[best_index:])
        
        # Intersection point
        vt1_vo2_threshold = (intercept2 - intercept1) / (slope1 - slope2)
        vt1_vco2_threshold = slope1 * vt1_vo2_threshold + intercept1

        # Fit piecewise model with 2 segments (1 breakpoint)
        my_pwlf = pwlf.PiecewiseLinFit(vco2, ve)
        breaks = my_pwlf.fit(2)  # This returns x-values of breakpoints

        # VT2 is the breakpoint between segment 1 and 2
        vt2_vco2 = breaks[1]
        vt2_ve = my_pwlf.predict([vt2_vco2])[0]
        # Create plot
        if plot:
            fig, axs = plt.subplots(1, 2, figsize=(14, 6))
            # --- VT1 Plot ---
            axs[0].scatter(vo2, vco2, label='Data', color='lightgray')
            axs[0].plot(vo2[:best_index], slope1 * vo2[:best_index] + intercept1, 'b', label='Slope < 1')
            axs[0].plot(vo2[best_index:], slope2 * vo2[best_index:] + intercept2, 'r', label='Slope ≥ 1')
            axs[0].scatter(vt1_vo2_threshold, vt1_vco2_threshold, color='black', zorder=3, label='VT1')
            axs[0].set_xlabel('VO₂ (L/min)')
            axs[0].set_ylabel('VCO₂ (L/min)')
            axs[0].set_title('V-Slope Method (VT1)')
            axs[0].legend()
            axs[0].grid()

               # --- VT2 Plot using PWLF ---
            axs[1].scatter(vco2, ve, label='Data', color='lightgray')

            # Plot the fitted segments
            x_hat = np.linspace(min(vco2), max(vco2), 100)
            y_hat = my_pwlf.predict(x_hat)
            axs[1].plot(x_hat, y_hat, 'b-', label='PWLF Fit')

            # Mark VT2
            axs[1].scatter(vt2_vco2, vt2_ve, color='black', marker='^', zorder=3, label='VT2 (PWLF)')
            axs[1].axvline(vt2_vco2, color='black', linestyle='--', alpha=0.5)

            axs[1].set_xlabel('VCO₂ (L/min)')
            axs[1].set_ylabel('VE (L/min)')
            axs[1].set_title('Ventilatory Compensation Point (VT2)')
            axs[1].legend()
            axs[1].grid()
            plt.tight_layout()
            plt.show()
                
        return {"vt1_vo2_level_vo2_vo2":vt1_vo2_threshold, "vt1_vo2_level_vo2_vco2":vt1_vco2_threshold,
                'vt2_vco2': vt2_vco2, 'vt2_ve': vt2_vco2}
    
    def vt1_vt2_VE(self, window_size, plot=False):
        VE_VO2 = self.df['VE/VO2'].rolling(f'{window_size}s', min_periods=1).mean()
        VE_CO2 = self.df['VE/VCO2'].rolling(f'{window_size}s', min_periods=1).mean()

        # Convert time index to total seconds
        time = self.df.index.total_seconds()
        # Identify nadir: Minimum VE/VO2 value
        nadir_index = VE_VO2.idxmin()
        if pd.isna(nadir_index) or nadir_index not in self.df.index:
            raise ValueError("nadir_index is NaN or not in DataFrame index")

        nadir_time = self.df.index[self.df.index.get_loc(nadir_index)]  # Safer method to get position
        nadir_value = VE_VO2[nadir_index]

        # Find the first rise after nadir while VE/CO2 is constant or increasing
        rise_index = None

        nadir_seconds = nadir_time.total_seconds()  # Convert Timedelta to seconds
        end_seconds = self.df.index[-1].total_seconds()
        for index in VE_CO2.index[1:]:
            prev_index = VE_CO2.index[VE_CO2.index < index].max()
            if VE_CO2.loc[index] >= VE_CO2.loc[prev_index] and VE_VO2.loc[index] > nadir_value:
                rise_index = prev_index
                break

        if rise_index is not None:
            rise_time = rise_index.total_seconds()
            rise_VE_VO2 = VE_VO2[rise_index]
            rise_VE_CO2 = VE_CO2[rise_index]
            #print(f"First rise after nadir found at time {rise_time} sec with VE/VO2 = {rise_VE_VO2} and VE/CO2 = {rise_VE_CO2}")
        else:
            print("No first rise after nadir found.")
        # Find the deflection point of VE/CO2 after nadir_index
        deflection_index = None
        for index in VE_CO2.index[VE_CO2.index > nadir_index]:
            if (index.total_seconds() - nadir_seconds) < 100:
                continue  # Ensure at least 5 seconds have passed
            prev_index = VE_CO2.index[VE_CO2.index < index].max()
            next_index = VE_CO2.index[VE_CO2.index > index].min()
            
            if prev_index is not None and next_index is not None:
                prev_slope = VE_CO2.loc[index] - VE_CO2.loc[prev_index]
                next_slope = VE_CO2.loc[next_index] - VE_CO2.loc[index]
                
                if prev_slope > 0 and next_slope < 0:  # Detect peak or deflection
                    deflection_index = index
                    break

        if deflection_index is not None:
            deflection_time = deflection_index.total_seconds()
            deflection_VE_CO2 = VE_CO2[deflection_index]
            #print(f"Deflection point of VE/CO2 found at time {deflection_time} sec with VE/CO2 = {deflection_VE_CO2}")
        else:
            print("No deflection point of VE/CO2 found after nadir.")
        vt1_speed, vt1_grade = self.get_speed_grade_from_time(nadir_time)
        vt2_speed, vt2_grade = self.get_speed_grade_from_time(deflection_index) if deflection_index is not None else (None, None)

        if plot:
            fig, ax = plt.subplots(figsize=(8, 6))

            ax.plot(time, VE_VO2, color='b', label='VE/VO2', zorder=2)
            ax.plot(time, VE_CO2, color='r', label='VE/CO2', zorder=1)
            ax.set_xlabel('Time (s)')
            ax.set_ylabel('Value')
            ax.tick_params(axis='y')

            # Mark the nadir, first rise, and deflection points
            ax.scatter(nadir_seconds, nadir_value, color='black', zorder=3, label='Nadir')
            if rise_index is not None:
                ax.scatter(rise_time, rise_VE_VO2, color='green', zorder=3, label='First Rise')
            if deflection_index is not None:
                ax.scatter(deflection_time, deflection_VE_CO2, color='purple', zorder=3, label='Deflection Point')

            fig.legend(loc='upper right', bbox_to_anchor=(0.9, 0.9))
            ax.grid(True)
            plt.title('VE/VO2 and VE/CO2 vs. Time')
            plt.tight_layout()
            plt.show()

        return {'vt1_time_VE': nadir_time, 'vt1_speed_VE': vt1_speed, 'vt1_grade_VE': vt1_grade,
                'vt2_time_VE': deflection_index if deflection_index is not None else None, 'vt2_speed_VE': vt2_speed, 'vt2_grade_VE': vt2_grade}

    def vt1_vt2_pet(self, window_size, plot=False):
            # Apply rolling mean
        PetO2 = self.df['PetO2'].rolling(f'{window_size}s', min_periods=1).mean()
        PetCO2 = self.df['PetCO2'].rolling(f'{window_size}s', min_periods=1).mean()
        
        # Convert time index to total seconds
        time = self.df.index.total_seconds()
        # Identify nadir: Minimum PetO2 value
        nadir_index = PetO2.idxmin()
        if pd.isna(nadir_index) or nadir_index not in self.df.index:
            raise ValueError("nadir_index is NaN or not in DataFrame index")

        nadir_time = self.df.index[self.df.index.get_loc(nadir_index)]  # Safer method to get position
        nadir_value = PetO2[nadir_index]

        # Find the first rise after nadir while PetCO2 is constant or increasing
        rise_index = None

        nadir_seconds = nadir_time.total_seconds()  # Convert Timedelta to seconds
        end_seconds = self.df.index[-1].total_seconds()
        for index in PetCO2.index[1:]:
            prev_index = PetCO2.index[PetCO2.index < index].max()
            if PetCO2.loc[index] >= PetCO2.loc[prev_index] and PetO2.loc[index] > nadir_value:
                rise_index = prev_index
                break

        if rise_index is not None:
            rise_time = rise_index.total_seconds()
            rise_PetO2 = PetO2[rise_index]
            rise_PetCO2 = PetCO2[rise_index]
            #print(f"First rise after nadir found at time {rise_time} sec with PetO2 = {rise_PetO2} and PetCO2 = {rise_PetCO2}")
        else:
            print("No first rise after nadir found.")
        # Compute the first derivative (rate of change) --> after aerobic threshold
        mask = PetCO2.index > nadir_index
        PetCO2_post_nadir = PetCO2[mask]
        time_post_nadir = time[mask]

        # Derivatives
        dPetCO2_dt = np.gradient(PetCO2_post_nadir, time_post_nadir)
        d2PetCO2_dt2 = np.gradient(dPetCO2_dt, time_post_nadir)

        # Deflection: point of max acceleration after nadir
        deflection_idx_local = np.argmax(d2PetCO2_dt2)
        deflection_index = PetCO2_post_nadir.index[deflection_idx_local]
        deflection_time = deflection_index
        deflection_value = PetCO2_post_nadir.iloc[deflection_idx_local]
        # Print results
        #print(f"Deflection point found at time {deflection_time} sec with PetCO2 = {deflection_value}")
        # Plot PetO2 and PetCO2
        vt1_speed, vt1_grade = self.get_speed_grade_from_time(nadir_index)
        vt2_speed, vt2_grade = self.get_speed_grade_from_time(deflection_time) if deflection_index is not None else (None, None)
        if plot:
            fig, ax1 = plt.subplots(figsize=(8, 6))

            ax1.plot(time, PetO2, color='b', label='PetO2', zorder=2)
            ax1.set_xlabel('Time (s)')
            ax1.set_ylabel('PetO2 (L/min)', color='b')
            ax1.tick_params(axis='y', labelcolor='b')

            ax2 = ax1.twinx()
            ax2.plot(time, PetCO2, color='r', label='PetCO2', zorder=1)
            ax2.set_ylabel('PetCO2 (L/min)', color='r')
            ax2.tick_params(axis='y', labelcolor='r')

            # Mark the nadir, first rise, and deflection points
            ax1.scatter(nadir_index.total_seconds(), nadir_value, color='black', zorder=3, label='Nadir')
            if rise_index is not None:
                ax1.scatter(rise_time, rise_PetO2, color='green', zorder=3, label='First Rise')
            ax2.scatter(deflection_time.total_seconds(), deflection_value, color='purple', zorder=3, label='Deflection Point')

            fig.legend(loc='upper right', bbox_to_anchor=(0.9, 0.9))
            ax1.grid(True)
            plt.title('PetO2 and PetCO2 vs. Time')
            plt.tight_layout()
            plt.show()

        return {'vt1_time_pet':nadir_time, 'vt1_speed_pet':vt1_speed, 'vt1_grade_pet':vt1_grade,
                'vt2_time_pet':deflection_time, 'vt2_speed_pet':vt2_speed, 'vt2_grade_pet':vt2_grade}
    
    def get_VO2_peak_time_metrics(self, window_size=30, rq_threshold=1.0):
        """
        Identifies the peak VO2 over a rolling time window and extracts related physiological metrics.

        Args:
            window_size (int): Duration of the rolling window in seconds (default: 30).
            rq_threshold (float): Threshold value of RQ to determine if true VO2max is achieved.

        Returns:
            dict: Dictionary containing VO2 peak value, its time window, and relevant physiological markers.
        """
        # Compute rolling average VO2 over the specified window
        rolling_avg = self.df['VO2'].rolling(f'{window_size}s').mean()

        # Identify the highest average VO2 and corresponding end time
        max_avg = rolling_avg.max()
        end_time = rolling_avg.idxmax()
        start_time = end_time - pd.Timedelta(seconds=30)

        # Handle case where no peak is detected
        if pd.isna(max_avg) or pd.isna(start_time) or pd.isna(end_time):
            return {"VO2_peak": None, "start_time": None, "end_time": None}

        # Retrieve grade, speed, and stage at end time, or fall back to start time if missing
        try:
            if pd.isna(self.df.loc[end_time, 'Stage']):
                grade = self.df.loc[start_time, 'Grade']
                speed = self.df.loc[start_time, 'Speed']
                stage = self.df.loc[start_time, 'Stage']
            else:
                grade = self.df.loc[end_time, 'Grade']
                speed = self.df.loc[end_time, 'Speed']
                stage = self.df.loc[end_time, 'Stage']
        except KeyError:
            # If exact end_time not found, find nearest index
            nearest_end_idx = self.df.index.get_indexer([end_time], method='nearest')[0]
            end_time = self.df.index[nearest_end_idx]
            start_time = end_time - pd.Timedelta(seconds=30)
            grade = self.df.loc[end_time, 'Grade']
            speed = self.df.loc[end_time, 'Speed']
            stage = self.df.loc[end_time, 'Stage']

        # Slice data within the peak window
        peak_window_df = self.df.loc[start_time:end_time]

        # Compute average physiological values within window
        rq_peak = peak_window_df['RQ'].mean()
        hr_peak = peak_window_df['HR'].mean()
        eem_peak = peak_window_df['EEm'].mean()
        fat_pct_peak = peak_window_df['Fat'].mean()
        cho_pct_peak = peak_window_df['CHO'].mean()
        vo2_kg_peak = peak_window_df['VO2/Kg'].mean()

        # Get first available lactate value after peak or last before if none exist
        post_peak = self.df.loc[self.df.index >= end_time]
        pre_peak = self.df.loc[self.df.index < end_time]
        if not post_peak['La-'].dropna().empty:
            lactate_post = post_peak['La-'].dropna().iloc[0]
        else:
            lactate_post = pre_peak['La-'].dropna().iloc[-1] if not pre_peak['La-'].dropna().empty else None

        # Get first non-null RPE value after VO2 peak
        dyspnea_value = post_peak['Dyspnea'].dropna().iloc[0] if not post_peak['Dyspnea'].dropna().empty else None
        rpe_post = int(re.search(r'_(\d+)', dyspnea_value).group(1)) if dyspnea_value else None

        # Determine if this is a "true" VO2 max (based on RQ threshold)
        true_vo2max = rq_peak > rq_threshold if not pd.isna(rq_peak) else False

        return {
            "VO2_peak": max_avg,
            "VO2_peak_start_time": start_time,
            "VO2_peak_end_time": end_time,
            "GradePeak": grade,
            "SpeedPeak": speed,
            "MarkerPeak": stage,
            "RQPeak": rq_peak,
            "HRPeak": hr_peak,
            "EEMPeak": eem_peak,
            "Fat%Peak": fat_pct_peak,
            "CHO%Peak": cho_pct_peak,
            "VO2/kgPeak": vo2_kg_peak,
            "Lactate-VO2Peak": lactate_post,
            "RPE-VO2peak": rpe_post,
            "True_VO2max": true_vo2max
        }

    def get_all_metrics(self, vo2_window_size=50, vt_window_size=30, rq_threshold=1.0, plot=False):
        """
        Executes all relevant metric extraction functions and compiles the results.

        Args:
            vo2_window_size (int): Window size for VO2 peak calculation.
            vt_window_size (int): Window size for ventilatory threshold functions.
            rq_thresh (float): RQ threshold to determine true VO2max.

        Returns:
            dict: Aggregated dictionary of all computed metrics.
        """
        funcs_with_args = {
            'get_VO2_peak_time_metrics': {'window_size': vo2_window_size, 'rq_threshold': rq_threshold},
            'lt_abs_vals': {'plot': plot},
            'lt_ref_vals': {'plot': plot},
            'lt_log_semilog': {'plot': plot},
            'vt1_vt2_vo2_vco2': {'plot': plot},
            'vt1_vt2_VE': {'window_size': vt_window_size, 'plot': plot},  
            'vt1_vt2_pet': {'window_size': vt_window_size, 'plot': plot},
        }

        results = {}
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", category=RuntimeWarning)
            for func_name, kwargs in funcs_with_args.items():
                try:
                    method = getattr(self, func_name)
                    result = method(**kwargs)
                    if isinstance(result, dict):
                        results.update(result)
                    else:
                        results[func_name] = result
                except Exception as e:
                    results[f"{func_name}_error"] = str(e)
        return results
    
    def get_speed_grade_from_time(self, time):
        #time = pd.to_timedelta(time)

        # Slice everything up to and including 'time'
        sub_df = self.df.loc[:time]
        if sub_df.empty:
            raise ValueError(f"No data available at or before time {time}")

        row = sub_df.iloc[-1]  # Last row before or equal to time
        return row['Speed'], row['Grade']

    def is_steady_state(self, O2_series, steady_state_threshold=0.02):
        """
        Determines if VO2 is in steady state based on coefficient of variation.

        Args:
            O2_series (pd.Series): Time series of VO2 values.
            threshold (float): Max coefficient of variation for steady state.

        Returns:
            bool: True if variation is below threshold, otherwise False.
        """
        variance = O2_series.std() / O2_series.mean()
        return variance < steady_state_threshold

    def get_stage_metrics(self, stage_df, summary_df, window_size_seconds, rq_threshold, smoothing_window, steady_state_threshold):
        """
        Extracts summary statistics for a given test stage and appends them to the summary DataFrame.

        Args:
            stage_df (pd.DataFrame): Subset of main data for a single stage.
            summary_df (pd.DataFrame): DataFrame where metrics are recorded.
            window_size_seconds (int): Duration of time window before stage ends.
            rq_threshold (float): Threshold for classifying true VO2max.
            smoothing_window (int): Window size for rolling mean calculations for steady state.
        """
        if stage_df.empty:
            print(f"Warning: Empty DataFrame for stage {stage_df.name}")
            return

        # Handle various index formats for window slicing
        if isinstance(stage_df.index, pd.TimedeltaIndex):
            base_time = pd.Timestamp('1970-01-01')
            datetime_index = base_time + stage_df.index
            end_time_dt = datetime_index[-1]
            start_time_dt = end_time_dt - pd.Timedelta(seconds=window_size_seconds)
            window_df = stage_df[(datetime_index >= start_time_dt)]
            start_time = start_time_dt - base_time
            end_time = end_time_dt - base_time
        elif not isinstance(stage_df.index, pd.DatetimeIndex):
            stage_df = stage_df.set_index(pd.to_datetime(stage_df.index))
            end_time = stage_df.index[-1]
            start_time = end_time - pd.Timedelta(seconds=window_size_seconds)
            window_df = stage_df[stage_df.index >= start_time]
        else:
            end_time = stage_df.index[-1]
            start_time = end_time - pd.Timedelta(seconds=window_size_seconds)
            window_df = stage_df[stage_df.index >= start_time]

        if window_df.empty:
            return  # Skip if not enough data in the window

        # Initialize new row for current stage
        summary_df.loc[stage_df.name] = np.nan
        summary_df.loc[stage_df.name, 'Analysis Start Time'] = str(start_time).split('.')[0]
        summary_df.loc[stage_df.name, 'Analysis End Time'] = str(end_time).split('.')[0]

        # Fill in core metrics for the stage
        summary_df.loc[stage_df.name, 'Velocity (km/h)'] = stage_df['Speed'].iloc[0]
        summary_df.loc[stage_df.name, 'Grade (%)'] = stage_df['Grade'].iloc[0]
        summary_df.loc[stage_df.name, 'Stage (#)'] = stage_df['Stage'].iloc[0]

        # Convert RPE string to integer if needed
        last_rpe = stage_df['Dyspnea'].iloc[-1]
        if isinstance(last_rpe, str):
            summary_df.loc[stage_df.name, 'RPE (#)'] = int(last_rpe.split("_")[-1])
        else:
            summary_df.loc[stage_df.name, 'RPE (#)'] = last_rpe

        # Retrieve last available lactate value
        last_lactate = stage_df['La-'].dropna().iloc[-1] if not stage_df['La-'].dropna().empty else np.nan
        summary_df.loc[stage_df.name, 'Lactate (mmol/L)'] = last_lactate

        # Calculate mean physiological measures over the time window
        summary_df.loc[stage_df.name, 'VO2 (mL/min)'] = window_df['VO2'].mean()
        summary_df.loc[stage_df.name, 'VO2 (mL/min/kg)'] = window_df['VO2/Kg'].mean()
        summary_df.loc[stage_df.name, 'HR (bpm)'] = window_df['HR'].mean()
        summary_df.loc[stage_df.name, 'RQ'] = window_df['RQ'].mean()

        # Classify VO2max and check for steady state
        summary_df.loc[stage_df.name, 'True VO2max? T/F'] = summary_df.loc[stage_df.name, 'RQ'] > rq_threshold
        vo2_rolling = window_df['VO2'].rolling(f'{smoothing_window}s', min_periods=1).mean()
        summary_df.loc[stage_df.name, 'Reach Steady State? T/F'] = self.is_steady_state(vo2_rolling, steady_state_threshold)

    def create_test_summary(self, window_size_seconds=50, rq_threshold=1.0, smoothing_window=3, steady_state_threshold=0.02):
        """
        Generates a summary DataFrame containing stage-wise metrics.

        Args:
            window_size_seconds (int): Duration of time window for metric calculation before each stage end.
            rq_threshold (float): RQ threshold used to flag true VO2max values.

        Returns:
            pd.DataFrame: A summary table with metrics for each stage.
        """
        summary_df = pd.DataFrame({
            'Analysis Start Time': pd.Series(dtype='str'),
            'Analysis End Time': pd.Series(dtype='str'),
            'Velocity (km/h)': pd.Series(dtype='float'),
            'Grade (%)': pd.Series(dtype='float'),
            'Stage (#)': pd.Series(dtype='float'),
            'RPE (#)': pd.Series(dtype='float'),
            'Lactate (mmol/L)': pd.Series(dtype='float'),
            'VO2 (mL/min)': pd.Series(dtype='float'),
            'VO2 (mL/min/kg)': pd.Series(dtype='float'),
            'HR (bpm)': pd.Series(dtype='float'),
            'RQ': pd.Series(dtype='float'),
            'True VO2max? T/F': pd.Series(dtype='bool'),
            'Reach Steady State? T/F': pd.Series(dtype='bool'),
        })

        # Loop over each unique stage and calculate metrics
        for stage_name, group in self.df.groupby('Stage'):
            if stage_name == 0:
                continue  # Skip Stage 0
            group.name = stage_name
            self.get_stage_metrics(group, summary_df, window_size_seconds, rq_threshold,smoothing_window, steady_state_threshold)

        return summary_df
